# figure2.ipynb

Manuscript analysis and plotting notebook.

Run the configuration cell first. Input data and fonts are not bundled.
See the repository README and data requirements before executing.


In [ ]:
from pathlib import Path
import sys as _sys

_repo_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / 'plot_config.py').is_file()
     and (p / 'config.example.json').is_file()),
    None,
)
if _repo_root is None:
    raise RuntimeError('Run this notebook from within the plotting repository.')
if str(_repo_root) not in _sys.path:
    _sys.path.insert(0, str(_repo_root))
from plot_config import configured_path


In [ ]:
import numpy as np  
import matplotlib.pyplot as plt  
import matplotlib as mpl
from matplotlib import font_manager
from pathlib import Path
import matplotlib.patches as patches 
import xarray as xr
import pandas as pd
import cartopy.crs as ccrs  
import cartopy.feature as cfeature
import cmaps
from cartopy.util import add_cyclic_point
import matplotlib.ticker as ticker
import matplotlib.dates as mdates

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

times_font_dir = Path(configured_path('font_dir'))
times_font_files = [times_font_dir / 'Times New Roman.ttf', times_font_dir / 'Times New Roman Bold.ttf']
for font_file in times_font_files:
    if not font_file.is_file():
        raise FileNotFoundError(f'Required Times New Roman font not found: {font_file}')
    font_manager.fontManager.addfont(str(font_file))
resolved_times = Path(font_manager.findfont('Times New Roman', fallback_to_default=False)).resolve()
if resolved_times not in {font.resolve() for font in times_font_files}:
    raise RuntimeError(f'Times New Roman resolved to an unexpected file: {resolved_times}')
mpl.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'mathtext.fontset': 'custom',
    'mathtext.rm': 'Times New Roman',
    'mathtext.it': 'Times New Roman:italic',
    'mathtext.bf': 'Times New Roman:bold',
})

In [ ]:
# 使用已划分的训练集和验证集，无需额外读取海洋掩码。

In [ ]:
train_file = Path(configured_path('training_pairs_file'))
val_file = Path(configured_path('validation_pairs_file'))
ds_train = xr.open_dataset(train_file)
ds_val = xr.open_dataset(val_file)

In [ ]:
display(ds_train, ds_val)

In [ ]:
def prepare_mld_data(ds):
    """
    从数据集提取MLD观测值和预测值，并去除NaN值
    
    参数:
    ds: xarray.Dataset，包含'MLD_true'和'MLD_pred'变量
    
    返回:
    tuple: (obs, preds) - 观测值和预测值的数组
    """
    # 获取真实MLD值
    mask_t = ds.MLD_true
    
    # 对预测值进行掩码处理，只保留有观测值的区域
    pred = ds.MLD_pred.where(~np.isnan(mask_t))
    
    # 提取数组值
    obs = ds.MLD_true.values
    preds = pred.values
    
    # 创建掩码，去除NaN值
    mask = np.isfinite(obs) & np.isfinite(preds)
    
    # 应用掩码
    obs = obs[mask]
    preds = preds[mask]
    
    return obs, preds


In [ ]:
obs1, preds1 = prepare_mld_data(ds_train)
obs2, preds2 = prepare_mld_data(ds_val)

In [ ]:
# 直接使用文件中已有的划分，不再随机混合或重新拆分样本。
print(f'Training: {len(obs1):,} valid observation/prediction pairs')
print(f'Validation: {len(obs2):,} valid observation/prediction pairs')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

def plot_hexbin_comparison(obs1, preds1, obs2, preds2, 
                          labels=('(a)', '(b)'), 
                          titles=('Model 1', 'Model 2'),
                          xlim=(0, 600), ylim=(0, 600),
                          figsize=(13, 6),
                          hex_gridsize=180):
    """
    绘制两个hexbin对比图，共用colorbar
    观测值作为x轴，预测值作为y轴
    
    参数:
    obs1, preds1: 第一组观测值和预测值
    obs2, preds2: 第二组观测值和预测值
    labels: 子图标签，默认为('(a)', '(b)')
    titles: 子图标题，默认为('Model 1', 'Model 2')
    xlim, ylim: 坐标轴范围
    figsize: 图形大小
    hex_gridsize: hexbin的网格大小，值越大点越小
    """
    
    # 计算统计指标
    def calculate_metrics(obs, preds):
        r2 = r2_score(obs, preds)
        rmse = np.sqrt(mean_squared_error(obs, preds))
        mae = mean_absolute_error(obs, preds)
        mbe = np.mean(preds - obs)  # 注意：这里改为预测值减去观测值
        n_points = len(obs)
        return r2, rmse, mae, mbe, n_points
    
    # 计算两组数据的统计指标
    r2_1, rmse_1, mae_1, mbe_1, n_points_1 = calculate_metrics(obs1, preds1)
    r2_2, rmse_2, mae_2, mbe_2, n_points_2 = calculate_metrics(obs2, preds2)
    
    # 创建图形和子图
    fig, axes = plt.subplots(1, 2, figsize=figsize, sharex=True, sharey=True,
                              constrained_layout=False)
    all_hexbins = []
    
    # 使用完整数据绘制对数计数 hexbin，保留高密度散点结构
    
    # 绘制第一个子图（观测值在x轴，预测值在y轴）
    ax1 = axes[0]
    hb1 = ax1.hexbin(obs1, preds1, gridsize=hex_gridsize, cmap='RdYlBu_r',
                     bins='log', mincnt=1, linewidths=0, edgecolors='none',
                     alpha=0.95, extent=(*xlim, *ylim), rasterized=True)
    all_hexbins.append(hb1)
    
    # 添加参考线 y=x
    ax1.plot(xlim, ylim, color='gray', linestyle='-', 
             linewidth=1.5, alpha=0.8)
    
    # 添加统计信息 - 增大字体
    textstr1 = '\n'.join((
        f'$R^2$ = {r2_1:.2f}',
        f'RMSE = {rmse_1:.2f} m',
        f'MAE = {mae_1:.2f} m',
        f'MBE = {mbe_1:.2f} m',
        f'N = {n_points_1:,}'
    ))
    ax1.text(0.05, 0.95, textstr1, transform=ax1.transAxes,
            fontsize=15, verticalalignment='top', horizontalalignment='left',
            bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', 
                     boxstyle='round,pad=0.3'))
    
    # 坐标轴设置 - 增大字体
    ax1.set_xlabel('Observed MLD (m)', fontsize=16)
    ax1.set_ylabel('Predicted MLD (m)', fontsize=16)
    ax1.set_title(f'{labels[0]} {titles[0]}', fontsize=17, fontweight='bold', loc='left', pad=8)
    ax1.set_xlim(xlim[0], xlim[1])
    ax1.set_xticks(np.arange(xlim[0], xlim[1] + 1, 100))
    ax1.set_ylim(ylim[0], ylim[1])
    ax1.set_aspect('equal')
    ax1.set_axisbelow(True)
    ax1.grid(True, linestyle='-', linewidth=0.7, color='0.82', alpha=0.8)
    ax1.tick_params(axis='both', which='major', labelsize=15)
    
    # 绘制第二个子图（观测值在x轴，预测值在y轴）
    ax2 = axes[1]
    hb2 = ax2.hexbin(obs2, preds2, gridsize=hex_gridsize, cmap='RdYlBu_r',
                     bins='log', mincnt=1, linewidths=0, edgecolors='none',
                     alpha=0.95, extent=(*xlim, *ylim), rasterized=True)
    all_hexbins.append(hb2)
    
    # 添加参考线 y=x
    ax2.plot(xlim, ylim, color='gray', linestyle='-', 
             linewidth=1.5, alpha=0.8)
    
    # 添加统计信息 - 增大字体
    textstr2 = '\n'.join((
        f'$R^2$ = {r2_2:.2f}',
        f'RMSE = {rmse_2:.2f} m',
        f'MAE = {mae_2:.2f} m',
        f'MBE = {mbe_2:.2f} m',
        f'N = {n_points_2:,}'
    ))
    ax2.text(0.05, 0.95, textstr2, transform=ax2.transAxes,
            fontsize=15, verticalalignment='top', horizontalalignment='left',
            bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', 
                     boxstyle='round,pad=0.3'))
    
    # 坐标轴设置 - 增大字体
    ax2.set_xlabel('Observed MLD (m)', fontsize=16)
    ax2.set_ylabel('')
    ax2.set_title(f'{labels[1]} {titles[1]}', fontsize=17, fontweight='bold', loc='left', pad=8)
    ax2.set_xlim(xlim[0], xlim[1])
    ax2.set_xticks(np.arange(xlim[0], xlim[1] + 1, 100))
    ax2.set_ylim(ylim[0], ylim[1])
    ax2.set_aspect('equal')
    ax2.set_axisbelow(True)
    ax2.grid(True, linestyle='-', linewidth=0.7, color='0.82', alpha=0.8)
    ax2.tick_params(axis='both', which='major', labelsize=15)
    
    # 调整布局
    plt.tight_layout()
    
    
    vmin = min(hb.get_array().min() for hb in all_hexbins)
    vmax = max(hb.get_array().max() for hb in all_hexbins)
    hb1.set_clim(vmin, vmax)
    hb2.set_clim(vmin, vmax)
    fig.subplots_adjust(right=0.88)
    cbar_ax = fig.add_axes([0.89, ax1.get_position().y0, 0.015, ax1.get_position().height])
    cb = fig.colorbar(hb1, cax=cbar_ax)
    cb.set_label('Grid count', fontsize=17)
    cb.ax.tick_params(labelsize=15)
    
    plt.show()
    
    return fig, axes

In [ ]:
from matplotlib.colors import BoundaryNorm, ListedColormap

# 离散分段色带：低格点数使用浅色，高格点数使用暖色
discrete_colors = ['#f4f8fb', '#dcecf3', '#bfd9e6', '#f6e8b2',
                   '#f6c979', '#ee9050', '#b2182b']
cmap_discrete = ListedColormap(discrete_colors, name='density_bands')
count_boundaries = np.array([1, 5, 20, 100, 300, 1000, 3000, 10000])
norm_discrete = BoundaryNorm(count_boundaries, cmap_discrete.N, clip=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 6), sharex=True, sharey=True)
mappables = []
for ax, observed, predicted, panel, title in zip(
    axes, [obs1, obs2], [preds1, preds2], ['(a)', '(b)'], ['Training', 'Validation']
):
    hb = ax.hexbin(
        observed, predicted, gridsize=90, bins=None, mincnt=3,
        cmap=cmap_discrete, norm=norm_discrete, linewidths=0,
        edgecolors='none', alpha=0.95, extent=(0, 600, 0, 600), rasterized=True
    )
    # 保留网格计数，仅将显示符号改成圆点（每点代表一个计数网格）
    centers = hb.get_offsets().copy()
    counts = hb.get_array().copy()
    hb.remove()
    circles = ax.scatter(centers[:, 0], centers[:, 1], c=counts,
                         s=18, marker='o', cmap=cmap_discrete, norm=norm_discrete,
                         linewidths=0, edgecolors='none', alpha=0.95, rasterized=True)
    mappables.append(circles)
    ax.plot([0, 600], [0, 600], color='gray', lw=1.5, alpha=0.8)
    ax.set_title(f'{panel} {title}', fontsize=19, fontweight='bold', loc='center', pad=8)
    ax.set_xlim(0, 600); ax.set_ylim(0, 600)
    ax.set_xticks(np.arange(0, 601, 100)); ax.set_yticks(np.arange(0, 601, 100))
    ax.set_aspect('equal'); ax.set_axisbelow(True)
    ax.grid(True, color='0.82', lw=0.7, alpha=0.8)
    ax.tick_params(labelsize=16)

axes[0].set_xlabel('Observed MLD (m)', fontsize=17)
axes[0].set_ylabel('Predicted MLD (m)', fontsize=17)
axes[1].set_xlabel('Observed MLD (m)', fontsize=17)

for ax, observed, predicted in zip(axes, [obs1, obs2], [preds1, preds2]):
    r2 = r2_score(observed, predicted)
    rmse = np.sqrt(mean_squared_error(observed, predicted))
    mae = mean_absolute_error(observed, predicted)
    mbe = np.mean(predicted - observed)
    ax.text(0.05, 0.95, f'$R^2$ = {r2:.2f}\nRMSE = {rmse:.2f} m\nMAE = {mae:.2f} m\nMBE = {mbe:.2f} m\nN = {len(observed):,}',
            transform=ax.transAxes, fontsize=16, va='top',
            bbox=dict(facecolor='white', alpha=0.72, edgecolor='none'))

fig.subplots_adjust(right=0.88, wspace=0.08)
cbar_ax = fig.add_axes([0.89, axes[0].get_position().y0, 0.015, axes[0].get_position().height])
cb = fig.colorbar(mappables[0], cax=cbar_ax, boundaries=count_boundaries,
                 ticks=count_boundaries, spacing='uniform', extend='max')
cb.set_label('Grid count', fontsize=18)
cb.ax.tick_params(labelsize=14)
output_dir = Path(configured_path('output_dir'))
output_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(output_dir / 'figure2.png', dpi=600, bbox_inches='tight', pad_inches=0)
plt.show()
